# Tutorial: Using alternative glacier outlines, DEMs, and WGMS observations for mass-balance calibration

In this tutorial, we rebuild the OGGM glacier directory for `Hintereisferner` using alternative input data. We replace the default glacier outline with a new inventory, use an alternative DEM, and use WGMS observations for mass-balance calibration.

The aim of this tutorial is **not** to provide a universal recipe for mass-balance recalibration. Such a recipe does not really exist: the appropriate calibration strategy depends on the climate product, glacier region, observations available, research question, etc.

Check this publication to understand more about MB calibration challenges:

- Schuster L, Rounce DR, Maussion F. Glacier projections sensitivity to temperature-index model choices and calibration strategies. Annals of Glaciology. 2023;64(92):293-308. doi:10.1017/aog.2023.57. [link](https://www.cambridge.org/core/journals/annals-of-glaciology/article/glacier-projections-sensitivity-to-temperatureindex-model-choices-and-calibration-strategies/EBE19247F4ADC7888EB59EBECB0140B1)

Instead, this tutorial illustrates how to ingest your own data into an OGGM workflow, and which parts of the workflow you need to rethink when changing input data. In particular, we will identify which steps are affected by changing the glacier outline, DEM, and mass-balance observations.

We divided the tutorial in 3 different experiments for the same glacier. 

- Experiment A: change outline only.
- Experiment B: change DEM only. 
- Experiment C: change MB calibration target: use insitu observations vs geodetic MB.

You can also see how this changes your initial state: 
- Experiment D: change dynamic initialization strategies - OGGM has a whole tutorial about this: [Dynamic spinup and dynamic melt_f calibration for past simulations](https://tutorials.oggm.org/stable/notebooks/tutorials/dynamical_spinup.html)

For simplicity (in all experiments), **we keep the same baseline climate product as in the OGGM preprocessing directory `W5E5`**. We also reuse the precipitation factor and temperature bias from that reference setup (`prcp_fac`, `temp_bias`), calibrating only the melt factor of the glacier `melt_f`. This is a practical simplification for the tutorial, not a general recommendation. These parameters can also be affected indirectly by changes in glacier geometry, especially if the new outline or DEM significantly changes the glacier hypsometry.

However, thinking on only one parameter to calibrate in the MB, allows us to focus on the mechanics of rebuilding the glacier directory and understanding which workflow steps need to be repeated or reconsidered when we change model inputs. 

## What changes when we use new input data?

- Changing the glacier outline means that we need to redo the GIS tasks, glacier masks, elevation-band flowlines, apparent mass balance, inversion, and present-time glacier initialisation.

- Changing the DEM means that we also need to redo the GIS tasks and flowlines, because the glacier elevation distribution, slopes, and elevation-band areas may change.

- Changing the WGMS observations means that we need to rethink the mass-balance calibration target. 

- Changing the climate product would require a more complete recalibration strategy, including precipitation factor and temperature bias. We do not cover this in this tutorial.

In [ ]:
import logging
import sys
import os
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile
import tarfile

from oggm import cfg, utils, workflow, tasks, DEFAULT_BASE_URL, graphics
from oggm.core import massbalance

log = logging.getLogger(__name__)

In [ ]:
cfg.initialize(logging_level="DEBUG")

The code below downloads the standard OGGM setup from a Level 4 preprocessing directory. This reference setup uses the configuration shown in the table below and it also comes with a calibrated MB and an initial state derived via the dynamic spin-up. 

In the following experiments, we will change **one component at a time**. The aim is to understand which parts of the OGGM workflow need to be repeated when an input changes, and which inputs have the strongest effect on the initial glacier state.


| Setup / experiment | Glacier outline | DEM | Climate | MB calibration target | Dynamic initialization |
|---|---|---|---|---|---|
| Level 4 reference setup | RGI v6 | COPDEM | `GSWP3_W5E5` | Hugonnet et al. geodetic MB | Level 4 default / preprocessed state |
| Experiment A: outline only | **User-provided outline** | COPDEM | `GSWP3_W5E5` | Hugonnet et al. geodetic MB | Same as reference setup |
| Experiment B: DEM only | RGI v6 | **User-provided DEM** | `GSWP3_W5E5` | Hugonnet et al. geodetic MB | Same as reference setup |
| Experiment C: calibration target | RGI v6 | COPDEM | `GSWP3_W5E5` | **WGMS in-situ MB observations** | Same as reference setup |
| Experiment D: dynamic initialization | RGI v6 | COPDEM | `GSWP3_W5E5` | Hugonnet et al. geodetic MB | **Alternative dynamic initialization strategy** |

The best way to test different configurations for the same glacier or group of glaciers is to create multipe working directories. This will get easier with future versions of OGGM but for this tutorial the quickes will be to create a working dir per experiment, this can add up to a lot of data if you are doing global simulations.  

## Level 4 Reference set up

In [ ]:
# Reference gdir only for prcp_fac and temp_bias
cfg.PATHS['working_dir'] = utils.gettempdir(
    dirname='OGGM-reference-prepro',
    reset=True
)

base_url = ('https://cluster.klima.uni-bremen.de/~oggm/'
                'gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup')

gdirs = workflow.init_glacier_directories(['RGI60-11.00897'],
                                             from_prepro_level=4,
                                             prepro_base_url=base_url,
                                             prepro_border=80,
                                             prepro_rgi_version='62',
                                             reset=True,
                                             force=True)

gdir_ref = gdirs[0]

mb_calib_ref = gdir_ref.read_json('mb_calib')
fixed_prcp_fac = mb_calib_ref['prcp_fac']
fixed_temp_bias = mb_calib_ref['temp_bias']
reference_melt_f = mb_calib_ref['melt_f']

In [ ]:
mb_calib_ref

In [ ]:
print('We also have global parameters, which we assume to be the same for all glaciers')
gdir_ref.read_json('mb_calib')['mb_global_params']

In [ ]:
os.listdir(gdir_ref.dir)

In [ ]:
rgi_year = int(gdir_ref.rgi_date)  # 2003
rgi_year

Level 4 comes with the present-day state of a glacier

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

gdir = gdir_ref

# Open diagnostics
f_historical = gdir.get_filepath('model_diagnostics', filesuffix='_historical')
with xr.open_dataset(f_historical) as ds:
    ds_historical_no_spinup = ds.load()

f_spinup_historical = gdir.get_filepath('model_diagnostics',
                                        filesuffix='_spinup_historical')
with xr.open_dataset(f_spinup_historical) as ds:
    ds_historical_spinup = ds.load()

# Create two-panel figure
fig, axes = plt.subplots(ncols=2, figsize=(14, 4), sharex=True)

# a) Area
ax = axes[0]
ds_historical_no_spinup.area_m2.plot(ax=ax, label='Default - No dynamic spinup')
ds_historical_spinup.area_m2.plot(ax=ax, label='Default - With dynamic spinup')
ax.axvline(rgi_year, linestyle='--', linewidth=1.2, label=f'RGI year ({rgi_year})')
ax.set_title('a) Glacier area evolution')
ax.set_ylabel('Area (m²)')
ax.set_xlabel('Year')
ax.legend()

# b) Volume
ax = axes[1]
ax.axvline(rgi_year, linestyle='--', linewidth=1.2, label=f'RGI year ({rgi_year})')
ds_historical_no_spinup.volume_m3.plot(ax=ax, label='Default - No dynamic spinup')
ds_historical_spinup.volume_m3.plot(ax=ax, label='Default - With dynamic spinup')

ax.set_title('b) Glacier volume evolution')
ax.set_ylabel('Volume (m³)')
ax.set_xlabel('Year')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
f_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_historical')
with xr.open_dataset(f_historical, group='fl_0') as ds:
    dg_historical_no_spinup = ds.load()

f_spinup_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_spinup_historical')
with xr.open_dataset(f_spinup_historical, group='fl_0') as ds:
    dg_historical_spin = ds.load()

year = 2005

dg_historical_no_spinup.ice_velocity_myr.sel(time=year).plot(label='Default calibration No spinup')
dg_historical_spin.ice_velocity_myr.sel(time=year).plot(label='Default calibration with dynamic spinup')

plt.title(f'Ice velocity along the flowline in {year}')
plt.ylabel('Ice velocity (m yr⁻¹)')
plt.legend()

## Experiment A: change Glacier Outline only.

In this case we are using a different inventory for simulating the Hintereisferner glacier in the Alps. This new inventory was take from 
[Diaconu, C.‐A., Zekollari, H., & Bamber,J. L. (2025)](https://agupubs.onlinelibrary.wiley.com/doi/epdf/10.1029/2025EA004197) and provides outlines for two years 2015 and 2023.

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "Lanzhou_workshop_oggm_tutorials/data"
print(DATA_DIR)

In [ ]:
outlines_fp = os.path.join(DATA_DIR, 'outlines_2015/inv_preds_calib.shp')
outline = gpd.read_file(outlines_fp)

In [ ]:
# Path to original RGI outlines.tar.gz inside the glacier directory
outline_tar = Path(gdir_ref.dir) / 'outlines.tar.gz'
# Extract and read
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    with tarfile.open(outline_tar, 'r:gz') as tar:
        tar.extractall(tmpdir)

    # Find the shapefile inside the archive
    shp_files = list(tmpdir.rglob('*.shp'))
    print(shp_files)
    outline_rgi = gpd.read_file(shp_files[0])

In [ ]:
outline

In [ ]:
outline_rgi

### Difference between the outlines

In [ ]:
from plot_tools import plot_domain_with_extra_outlines

fig, ax = plt.subplots(figsize=(8, 8))

plot_domain_with_extra_outlines(
    [gdir_ref],
    outlines=[outline, outline_rgi],
    outline_labels=["Custom outline", "RGI outline"],
    outline_colors=["green", "grey"],
    ax=ax,
)

ax.set_title("Hintereisferner outlines")
plt.show()

You can see that the outline in green 2015 is much less than the 2003 RGI outline. 

### Customised OGGM run 
Before running a workflow with this inventory make sure you have the following configuration. Each change from the main workflow below is needed if you plan to use your own inventor. 

Note that we re-calibrate MB and re compute the spin-up. To show you hwo these parameters and the initial state change from changing the boundary conditions. 

We will keep the same DEM from the preprocessing directory.

In [ ]:
gdir_ref.get_filepath('dem')

Before giving OGGM your outline you have to make it look like the RGI in terms of its attributes:

In [ ]:
rgidf_simple = utils.cook_rgidf(outline, o1_region='11')
rgidf_simple

Important keep the same **RGIId** so we can download observations from OGGM for this glacier

In [ ]:
rgidf_simple['RGIId'] = outline_rgi['RGIId'].values
rgidf_simple

In [ ]:
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS["border"] = 80

# OGGM flags that needs to be on for this tutorial and configuration
cfg.PARAMS["continue_on_error"] = True
cfg.PARAMS["use_compression"] = True
cfg.PARAMS["use_tar_shapefiles"] = True
cfg.PATHS["rgi_version"] = "62"
cfg.PARAMS["use_temp_bias_from_file"] = True
cfg.PARAMS["compress_climate_netcdf"] = False
cfg.PARAMS["store_model_geometry"] = True
cfg.PARAMS["store_fl_diagnostics"] = True

cfg.PATHS['dem_file'] = gdir_ref.get_filepath('dem')

# --- Custom directory: rebuilt from new GIS inputs ---
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='OGGM-user-outline',
                                            reset=True)

gdirs = workflow.init_glacier_directories(rgidf_simple, reset=True, force=True)

# this task adds the DEM and defines the local grid
workflow.execute_entity_task(tasks.define_glacier_region, gdirs, source='USER')

elevation_band_task_list = [
        tasks.simple_glacier_masks,
        tasks.elevation_band_flowline,
        tasks.fixed_dx_elevation_band_flowline,
        tasks.compute_downstream_line,
        tasks.compute_downstream_bedshape,
    ]

for task in elevation_band_task_list:
    workflow.execute_entity_task(task, gdirs);

### Check difference in Hypsometry

In [ ]:
def plot_hypsometry(gdir, label):
    fls = gdir.read_pickle('inversion_flowlines')
    h = np.concatenate([fl.surface_h for fl in fls])
    w = np.concatenate([fl.widths for fl in fls])
    area = w * gdir.grid.dx**2

    bins = np.arange(np.floor(h.min() / 50) * 50,
                     np.ceil(h.max() / 50) * 50 + 50,
                     50)

    hist, edges = np.histogram(h, bins=bins, weights=area)
    z = 0.5 * (edges[:-1] + edges[1:])

    plt.plot(hist * 1e-6, z, label=label)


plt.figure(figsize=(6, 6))

plot_hypsometry(gdir_ref, 'Reference preprocessing - from OGGM')
plot_hypsometry(gdirs[0], 'Custom outline')

plt.xlabel('Area per elevation band [km²]')
plt.ylabel('Elevation [m]')
plt.legend()
plt.title('Hypsometry comparison')
plt.show()

In [ ]:
print('Lets keep in mind the default calibration provided by level 4')
mb_calib_ref

In [ ]:
from oggm.core.massbalance import MonthlyTIModel

cfg.PARAMS['baseline_climate'] = cfg.PARAMS['baseline_climate']

# add climate data to gdir
workflow.execute_entity_task(tasks.process_climate_data, gdirs)

for gdir in gdirs:
            tasks.mb_calibration_from_scalar_mb(
                gdir,
                ref_mb=mb_calib_ref['reference_mb'],
                ref_mb_err=mb_calib_ref['reference_mb_err'],
                ref_period=cfg.PARAMS['geodetic_mb_period'],
                write_to_gdir=True,
                overwrite_gdir=True,
                calibrate_param1='melt_f',
                calibrate_param2=None,
                calibrate_param3=None,
                prcp_fac=mb_calib_ref['prcp_fac'],
                temp_bias=mb_calib_ref['temp_bias'],
                mb_model_class=MonthlyTIModel)


mb_calib_outline = gdir.read_json('mb_calib')

print('This is how the calibration changed')
mb_calib_outline

Note the change in `melt_f`!

In [ ]:
mb_calib_outline['melt_f'] - mb_calib_ref['melt_f'] 

In [ ]:
workflow.execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs)

out = workflow.calibrate_inversion_from_consensus(
    gdirs,
    apply_fs_on_mismatch=True,
    error_on_mismatch=True,  # if you are running many glaciers some might not work
    filter_inversion_output=True,  # this partly filters the overdeepening due to
    # the equilibrium assumption for retreating glaciers (see. Figure 5 of Maussion et al. 2019)
    volume_m3_reference=None,  # here you could provide your own total volume estimate in m3
)

# Finally create the dynamic flowlines
workflow.execute_entity_task(tasks.init_present_time_glacier, gdirs)

### Now the spin-up for initial state

In [ ]:
cfg.PARAMS['evolution_model'] = 'SemiImplicit'

y0 = gdirs[0].get_climate_info()['baseline_yr_0']
ye = gdirs[0].get_climate_info()['baseline_yr_1'] + 1

y0, ye

In [ ]:
workflow.execute_entity_task(
    tasks.run_from_climate_data,
    gdirs,
    ys=y0,
    ye=ye,
    output_filesuffix='_outline_no_spinup',
    store_fl_diagnostics=True,
)

In [ ]:
minimise_for = 'area'

dynamic_spinup_start_year = 1901

workflow.execute_entity_task(
    tasks.run_dynamic_melt_f_calibration,
    gdirs,
    err_dmdtda_scaling_factor=0.2,
    ys=y0,
    ye=ye,
    kwargs_run_function={'minimise_for': minimise_for,
                         'store_fl_diagnostics': True},
    ignore_errors=True,
    kwargs_fallback_function={'minimise_for': minimise_for,
                              'store_fl_diagnostics': True},
    output_filesuffix='_spinup_historical',
)

In [ ]:
os.listdir(gdir.dir)

In [ ]:
f_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_outline_no_spinup')
with xr.open_dataset(f_historical, group='fl_0') as ds:
    dg_historical_no_spinup = ds.load()

f_spinup_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_spinup_historical')
with xr.open_dataset(f_spinup_historical, group='fl_0') as ds:
    dg_historical_spin = ds.load()

year = 2005

dg_historical_no_spinup.ice_velocity_myr.sel(time=year).plot(label='No spinup')
dg_historical_spin.ice_velocity_myr.sel(time=year).plot(label='With spinup')

plt.title(f'Ice velocity along the flowline in {year} for new outline and dynamic spinup')
plt.ylabel('Ice velocity (m yr⁻¹)')
plt.legend()

## Experiment B: Different DEM 

In [ ]:
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS["border"] = 80

# OGGM flags that needs to be on for this tutorial and configuration
cfg.PARAMS["continue_on_error"] = True
cfg.PARAMS["use_compression"] = True
cfg.PARAMS["use_tar_shapefiles"] = True
cfg.PATHS["rgi_version"] = "62"
cfg.PARAMS["use_temp_bias_from_file"] = True
cfg.PARAMS["compress_climate_netcdf"] = False
cfg.PARAMS["store_model_geometry"] = True
cfg.PARAMS["store_fl_diagnostics"] = True

cfg.PATHS['dem_file'] = utils.get_demo_file('hef_srtm.tif')

# --- Custom directory: rebuilt from new GIS inputs ---
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='OGGM-user-dem',
                                            reset=True)


orig = utils.get_rgi_glacier_entities(['RGI60-11.00897'], version='62')

gdirs = workflow.init_glacier_directories(orig, reset=True, force=True)

# this task adds the DEM and defines the local grid
workflow.execute_entity_task(tasks.define_glacier_region, gdirs, source='USER')

elevation_band_task_list = [
        tasks.simple_glacier_masks,
        tasks.elevation_band_flowline,
        tasks.fixed_dx_elevation_band_flowline,
        tasks.compute_downstream_line,
        tasks.compute_downstream_bedshape,
    ]

for task in elevation_band_task_list:
    workflow.execute_entity_task(task, gdirs);

In [ ]:
def plot_hypsometry(gdir, label):
    fls = gdir.read_pickle('inversion_flowlines')
    h = np.concatenate([fl.surface_h for fl in fls])
    w = np.concatenate([fl.widths for fl in fls])
    area = w * gdir.grid.dx**2

    bins = np.arange(np.floor(h.min() / 50) * 50,
                     np.ceil(h.max() / 50) * 50 + 50,
                     50)

    hist, edges = np.histogram(h, bins=bins, weights=area)
    z = 0.5 * (edges[:-1] + edges[1:])

    plt.plot(hist * 1e-6, z, label=label)


plt.figure(figsize=(6, 6))

plot_hypsometry(gdir_ref, 'Reference preprocessing - from OGGM')
plot_hypsometry(gdirs[0], 'Custom outline')

plt.xlabel('Area per elevation band [km²]')
plt.ylabel('Elevation [m]')
plt.legend()
plt.title('Hypsometry comparison')
plt.show()

In [ ]:
from oggm.core.massbalance import MonthlyTIModel

cfg.PARAMS['baseline_climate'] = cfg.PARAMS['baseline_climate']

# add climate data to gdir
workflow.execute_entity_task(tasks.process_climate_data, gdirs)

for gdir in gdirs:
            tasks.mb_calibration_from_scalar_mb(
                gdir,
                ref_mb=mb_calib_ref['reference_mb'],
                ref_mb_err=mb_calib_ref['reference_mb_err'],
                ref_period=cfg.PARAMS['geodetic_mb_period'],
                write_to_gdir=True,
                overwrite_gdir=True,
                calibrate_param1='melt_f',
                calibrate_param2=None,
                calibrate_param3=None,
                prcp_fac=mb_calib_ref['prcp_fac'],
                temp_bias=mb_calib_ref['temp_bias'],
                mb_model_class=MonthlyTIModel)


mb_calib_dem = gdir.read_json('mb_calib')

print('This is how the calibration changed')
mb_calib_dem

In [ ]:
mb_calib_dem['melt_f'] - mb_calib_ref['melt_f'] 

In [ ]:
mb_calib_outline['melt_f'] - mb_calib_ref['melt_f'] 

In [ ]:
workflow.execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs)

out = workflow.calibrate_inversion_from_consensus(
    gdirs,
    apply_fs_on_mismatch=True,
    error_on_mismatch=True,  # if you are running many glaciers some might not work
    filter_inversion_output=True,  # this partly filters the overdeepening due to
    # the equilibrium assumption for retreating glaciers (see. Figure 5 of Maussion et al. 2019)
    volume_m3_reference=None,  # here you could provide your own total volume estimate in m3
)

# Finally create the dynamic flowlines
workflow.execute_entity_task(tasks.init_present_time_glacier, gdirs)

In [ ]:
cfg.PARAMS['evolution_model'] = 'SemiImplicit'

y0 = gdirs[0].get_climate_info()['baseline_yr_0']
ye = gdirs[0].get_climate_info()['baseline_yr_1'] + 1

y0, ye

In [ ]:
workflow.execute_entity_task(
    tasks.run_from_climate_data,
    gdirs,
    ys=y0,
    ye=ye,
    output_filesuffix='_dem_no_spinup',
    store_fl_diagnostics=True,
)

In [ ]:
minimise_for = 'area'

dynamic_spinup_start_year = 1901

workflow.execute_entity_task(
    tasks.run_dynamic_melt_f_calibration,
    gdirs,
    err_dmdtda_scaling_factor=0.2,
    ys=y0,
    ye=ye,
    kwargs_run_function={'minimise_for': minimise_for,
                         'store_fl_diagnostics': True},
    ignore_errors=True,
    kwargs_fallback_function={'minimise_for': minimise_for,
                              'store_fl_diagnostics': True},
    output_filesuffix='_spinup_historical',
)

In [ ]:
f_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_dem_no_spinup')
with xr.open_dataset(f_historical, group='fl_0') as ds:
    dg_historical_no_spinup = ds.load()

f_spinup_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_spinup_historical')
with xr.open_dataset(f_spinup_historical, group='fl_0') as ds:
    dg_historical_spin = ds.load()

year = 2005

dg_historical_no_spinup.ice_velocity_myr.sel(time=year).plot(label='No spinup')
dg_historical_spin.ice_velocity_myr.sel(time=year).plot(label='With spinup')

plt.title(f'Ice velocity along the flowline in {year} for different DEM and dynamic spinup')
plt.ylabel('Ice velocity (m yr⁻¹)')
plt.legend()

## Experiment C: change MB calibration target: use insitu observations vs geodetic MB.

In [ ]:
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS["border"] = 80

# OGGM flags that needs to be on for this tutorial and configuration
cfg.PARAMS["continue_on_error"] = True
cfg.PARAMS["use_compression"] = True
cfg.PARAMS["use_tar_shapefiles"] = True
cfg.PATHS["rgi_version"] = "62"
cfg.PARAMS["use_temp_bias_from_file"] = True
cfg.PARAMS["compress_climate_netcdf"] = False
cfg.PARAMS["store_model_geometry"] = True
cfg.PARAMS["store_fl_diagnostics"] = True

cfg.PATHS['dem_file'] = gdir_ref.get_filepath('dem')

# --- Custom directory: rebuilt from new GIS inputs ---
cfg.PATHS['working_dir'] = utils.gettempdir(dirname='OGGM-user-dem',
                                            reset=True)


orig = utils.get_rgi_glacier_entities(['RGI60-11.00897'], version='62')

gdirs = workflow.init_glacier_directories(orig, reset=True, force=True)

# this task adds the DEM and defines the local grid
workflow.execute_entity_task(tasks.define_glacier_region, gdirs, source='USER')

elevation_band_task_list = [
        tasks.simple_glacier_masks,
        tasks.elevation_band_flowline,
        tasks.fixed_dx_elevation_band_flowline,
        tasks.compute_downstream_line,
        tasks.compute_downstream_bedshape,
    ]

for task in elevation_band_task_list:
    workflow.execute_entity_task(task, gdirs);

In [ ]:
print(cfg.PARAMS['baseline_climate'])
cfg.PARAMS['baseline_climate'] = cfg.PARAMS['baseline_climate']

# add climate data to gdir
workflow.execute_entity_task(tasks.process_climate_data, gdirs)

In [ ]:
for gdir in gdirs: 
    mb = gdir.get_ref_mb_data()

mb[['ANNUAL_BALANCE']].plot(title='WGMS data: Hintereisferner');

In [ ]:
mb

In [ ]:
reference_mb = mb['ANNUAL_BALANCE'].mean()
ref_period = mb.index.values

In [ ]:
for gdir in gdirs:
    tasks.mb_calibration_from_scalar_mb(
        gdir,
        ref_mb=reference_mb,
        ref_mb_years=ref_period,
        write_to_gdir=True,
        overwrite_gdir=True,
        calibrate_param1='melt_f',
        calibrate_param2=None,
        calibrate_param3=None,
        prcp_fac=mb_calib_ref['prcp_fac'],
        temp_bias=mb_calib_ref['temp_bias'],
        mb_model_class=MonthlyTIModel)

mb_calib_wgms = gdir.read_json('mb_calib')

In [ ]:
mb_calib_wgms

In [ ]:
mb_calib_wgms['melt_f'] - mb_calib_ref['melt_f']

In [ ]:
workflow.execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs)

out = workflow.calibrate_inversion_from_consensus(
    gdirs,
    apply_fs_on_mismatch=True,
    error_on_mismatch=True,  # if you are running many glaciers some might not work
    filter_inversion_output=True,  # this partly filters the overdeepening due to
    # the equilibrium assumption for retreating glaciers (see. Figure 5 of Maussion et al. 2019)
    volume_m3_reference=None,  # here you could provide your own total volume estimate in m3
)

# Finally create the dynamic flowlines
workflow.execute_entity_task(tasks.init_present_time_glacier, gdirs)

In [ ]:
cfg.PARAMS['evolution_model'] = 'SemiImplicit'

y0 = gdirs[0].get_climate_info()['baseline_yr_0']
ye = gdirs[0].get_climate_info()['baseline_yr_1'] + 1

y0, ye

In [ ]:
workflow.execute_entity_task(
    tasks.run_from_climate_data,
    gdirs,
    ys=y0,
    ye=ye,
    output_filesuffix='_wgms_no_spinup',
    store_fl_diagnostics=True,
)

In [ ]:
workflow.execute_entity_task(
    tasks.run_dynamic_spinup,
    gdirs,
    spinup_start_yr=1979,
    ye=ye,
    minimise_for='area',
    output_filesuffix='_wgms_calib_dynamic_spinup',
    store_fl_diagnostics=True,
    ignore_errors=True,
)

In [ ]:
os.listdir(gdir.dir)

In [ ]:
f_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_wgms_no_spinup')
with xr.open_dataset(f_historical, group='fl_0') as ds:
    dg_historical_no_spinup = ds.load()

f_spinup_historical = gdir.get_filepath('fl_diagnostics', filesuffix='_wgms_calib_dynamic_spinup')
with xr.open_dataset(f_spinup_historical, group='fl_0') as ds:
    dg_historical_spin = ds.load()

year = 2005

dg_historical_no_spinup.ice_velocity_myr.sel(time=year).plot(label='No spinup')
dg_historical_spin.ice_velocity_myr.sel(time=year).plot(label='With spinup')

plt.title(f'Ice velocity along the flowline in {year} for WGMS calibration and standard spinup (no dynamic evolution)')
plt.ylabel('Ice velocity (m yr⁻¹)')
plt.legend()

In [ ]:
mb_calib_ref

In [ ]:
results = []

results.append({
        'case': 'Level 4, reference set up',
        'rgi_id': gdir.rgi_id,
        'melt_f': mb_calib_ref['melt_f'],
        'prcp_fac': mb_calib_ref['prcp_fac'],
        'temp_bias': mb_calib_ref['temp_bias'],
    })

results.append(
    {
        'case': 'Experiment A: different outline',
        'rgi_id': gdir.rgi_id,
        'melt_f': mb_calib_outline['melt_f'],
        'prcp_fac': mb_calib_outline['prcp_fac'],
        'temp_bias': mb_calib_outline['temp_bias'],
    })
results.append(
    {
        'case': 'Experiment B: different DEM',
        'rgi_id': gdir.rgi_id,
        'melt_f': mb_calib_dem['melt_f'],
        'prcp_fac': mb_calib_dem['prcp_fac'],
        'temp_bias': mb_calib_dem['temp_bias'],
    })
results.append(
    {
        'case': 'Experiment C: different observations for calibration',
        'rgi_id': gdir.rgi_id,
        'melt_f': mb_calib_wgms['melt_f'],
        'prcp_fac': mb_calib_wgms['prcp_fac'],
        'temp_bias': mb_calib_wgms['temp_bias'],
    })

In [ ]:
df_calib = pd.DataFrame(results)

df_calib

In [ ]:
ref_melt_f = mb_calib_ref['melt_f']
df_calib['melt_f_change_from_ref_%'] = (
    (df_calib['melt_f'] - ref_melt_f) / ref_melt_f * 100
)

df_calib

## Conclusion

- The largest change in the calibrated melt factor `melt_f` (+13.78%) occurs when we change the mass-balance calibration target from Hugonnet et al. geodetic mass balance to WGMS in-situ observations (Experiment C). For this glacier and setup, the WGMS calibration leads to a higher `melt_f`.

- Changing the outline and DEM also affects `melt_f`, because these inputs modify the glacier geometry, hypsometry, and elevation-band structure used by the mass-balance model.

**The key message is that calibration is part of the experiment design. If you change the input data in OGGM, you need to consider which workflow steps must be repeated, which parameters can be reused, and how the new calibration should be interpreted.**

- The velocity profiles between experiments, show that dynamic initialization can have very different effects on the start of our simulation (before a evolving with a climate scenario) depending on the calibration strategy. In experiment C, the no-spinup and spinup profiles are relatively similar, suggesting that the manually calibrated MB parameters (with WGMS observations) already produce an initial state close to the spun-up state. 

In contrast, Level 4 (the default/Hugonnet experiment with dynamic melt-factor calibration) shows a much larger difference. Here, the spinup procedure changes both the glacier geometry and the melt-factor calibration, leading to a substantially different ice-thickness and flux distribution. Since ice velocity is highly sensitive to thickness and slope, this produces a much larger change in the velocity profile.

This comparison shows that calibration and dynamic initialization are not independent choices: the calibration strategy can determine how much the spinup needs to modify the glacier state.

## Conclusion

- The largest change in the calibrated melt factor `melt_f` (+13.78%) occurs when we change the mass-balance calibration target; from Hugonnet et al. geodetic mass balance to WGMS in-situ observations (Experiment C). For this glacier and setup, the WGMS calibration leads to a higher `melt_f`.

- Changing the outline and DEM also affects `melt_f`, because these inputs modify the glacier geometry, hypsometry, and elevation-band structure used by the mass-balance model.

**The key message is that calibration is part of the experiment design. If you change the input data in OGGM, you need to consider which workflow steps must be repeated, which parameters can be reused, and how the new calibration should be interpreted.**

- The velocity profiles from the different experiments, show that dynamic initialization can have very different effects on the initial glacier state, before the glacier is evolved with a climate scenario. These effects depend on the calibration strategy. In Experiment C, the no-spinup and spinup profiles are relatively similar, suggesting that the manually calibrated MB parameters, based on WGMS observations, already produce an initial state close to the state after spinup. Changing the DEM and the outline does not affect much an initial state at least for this glacier.

- In contrast, Level 4, the default/Hugonnet experiment with dynamic melt-factor calibration, shows a much larger difference between the no-spinup and spinup velocity profiles. Here, the spinup procedure changes both the glacier geometry and the melt-factor calibration, leading to a substantially different ice-thickness and flux distribution. Since ice velocity is highly sensitive to thickness and slope, this produces a much larger change in the velocity profile. Which is actually needed it to reduce “initial numerical shocks” in the first years of the simulation.

This comparison shows that calibration and dynamic initialization are not independent choices: the calibration strategy **(and observations used)** can determine how much the spinup needs to modify the glacier state.

A natural next step is the sensitivity analysis tutorial, where you can explore parameter uncertainty and its effect on the simulations.

https://tutorials.oggm.org/stable/notebooks/tutorials/massbalance_perturbation.html